In [44]:
import urllib.parse

import sqlalchemy as sql
import sqlalchemy.orm as orm

db_path = r'D:\K\Desktop\DatabaseISSEP.accdb'
odbc_driver = "Microsoft Access Driver (*.mdb, *.accdb)"

connection_string = (
    f'DRIVER={{{odbc_driver}}};'
    f'DBQ={db_path};'
    'ReadOnly=0;'
    'ExtendedAnsiSQL=1;'
)

connection_url = sql.URL.create('access+pyodbc', query={
    'odbc_connect': urllib.parse.quote_plus(connection_string)
})

engine = sql.create_engine(url=connection_url)

session_factory = orm.sessionmaker(bind=engine)

In [78]:
from datetime import date


class Base(orm.DeclarativeBase):
    pass

class Livre(Base):
    __tablename__ = 'Livres'
    id: orm.Mapped[int] = orm.mapped_column(primary_key=True)
    titre: orm.Mapped[str] = orm.mapped_column()
    annee: orm.Mapped[int] = orm.mapped_column()
    date_achat: orm.Mapped[date] = orm.mapped_column()
    etranger: orm.Mapped[bool] = orm.mapped_column()
    auteur_id: orm.Mapped[int] = orm.mapped_column(sql.ForeignKey('Auteurs.id'))
    auteur: orm.Mapped['Auteur'] = orm.relationship()

class Auteur(Base):
    __tablename__ = 'Auteurs'
    id: orm.Mapped[int] = orm.mapped_column(primary_key=True)
    nom: orm.Mapped[str] = orm.mapped_column()
    prenom: orm.Mapped[str] = orm.mapped_column()
    livres: orm.Mapped[list['Livre']] = orm.relationship()

In [84]:
from operator import attrgetter

with session_factory() as session:
    query = sql.select(Auteur)
    data = session.scalars(query).all()
    for a in data:
        print(a.nom)
        for l in sorted(a.livres, key=attrgetter('titre')):
            print(l.titre)


Camus
La peste 42
Tolkien
Le retour du roi
Le seigneur des anneaux
Les 2 tours


In [ ]:
titre = input('Entrez le titre')
annee = input('Entrez l\'année')
etranger = input('Le livre est-il étranger')

livre = Livre(
    titre=titre,
    annee=int(annee),
    date_achat=date.today(),
    etranger=etranger=='o',
)

with session_factory() as session:
    session.add(livre)
    session.commit()

In [ ]:
with session_factory() as session:
    try:
        query = sql.select(Livre).where(Livre.id == 3)
        livre = session.scalars(query).one()
        livre.annee = 1999
        session.commit()
    except:
        print('le livre n\'existe plus')

le livre n'existe plus


In [62]:
with session_factory() as session:
    try:
        query = sql.select(Livre).where(Livre.id == 3)
        livre = session.scalars(query).one()
        session.delete(livre)
        session.commit()
    except:
        print('le livre n\'existe plus')

le livre n'existe plus


In [ ]:
with session_factory() as session:
    try:
        query = sql.select(Auteur).where(Auteur.id == 2)
        a = session.scalars(query).one()
        l = Livre(
            titre='Le retour du roi',
            annee=1975,
            etranger=False,
            date_achat=date(2026,9,4)
        )
        a.livres.append(l)
        session.commit()
    except:
        print('L\'auteur n\'existe plus')